In [2]:
###RAG Pipelines-Data Ingestion to Vector DB Pipeline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)
    #Find all pdf files recursively
    pdf_files =list(pdf_dir.glob("**/*.pdf"))
    print (f"found {len(pdf_files)} PDF files to process")
    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")
        except Exception as e:
            print(f" ❌Error {e}")
    print(f"\n all documents loaded")
    return all_documents

all_documents=process_all_pdfs("../data")




found2 PDF files to process

 Processing: ai_rag_notes.pdf
loaded 1 pages

 Processing: python_notes.pdf
loaded 1 pages

 all documents loaded


In [13]:
all_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T04:54:38+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T04:54:38+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\ai_rag_notes.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_rag_notes.pdf', 'file_type': 'pdf'}, page_content='Ai Rag Notes\nArtificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems that can perform\ntasks associated with human intelligence, such as reasoning, learning, perception, and language\nunderstanding.\nMachine Learning\nMachine Learning is a subset of AI in which algorithms learn patterns from data and use those patterns to\nmake predictions or decisions.\nGenerative AI\nGenerative AI systems create new content such as text, images, audio, video, and code. Large language\

In [ ]:
###Text splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"{len(documents)} documents coverted into {len(split_docs)} chunks")
   # example of chunks
    if split_docs:
     print(f"\n Example chunks")
     print(f"Contents:{split_docs[0].page_content[:100]}...")
     print(f"Metadata:{split_docs[0].metadata}")

    return split_docs



In [22]:
chunks=split_documents(all_documents)
chunks

2documents coverted into 3 chunks

 Example chunks
Contents:Ai Rag Notes
Artificial Intelligence
Artificial Intelligence (AI) is the field of computer science c...
Metadata:{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T04:54:38+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T04:54:38+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\ai_rag_notes.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_rag_notes.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T04:54:38+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T04:54:38+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\ai_rag_notes.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_rag_notes.pdf', 'file_type': 'pdf'}, page_content='Ai Rag Notes\nArtificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems that can perform\ntasks associated with human intelligence, such as reasoning, learning, perception, and language\nunderstanding.\nMachine Learning\nMachine Learning is a subset of AI in which algorithms learn patterns from data and use those patterns to\nmake predictions or decisions.\nGenerative AI\nGenerative AI systems create new content such as text, images, audio, video, and code. Large language\

###embedding and VectorStoreDB

In [23]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import Dict,Tuple,List,Any
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading embedding model {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Loaded embedding model successfully.Embedding dimensions {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise 
    def generate_embedding(self,texts:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for {len(texts)} texts ...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape {embeddings.shape}")
        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6163.69it/s]


Loaded embedding model successfully.Embedding dimensions 384


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6308\3300416845.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Loaded embedding model successfully.Embedding dimensions {self.model.get_sentence_embedding_dimension()}")


###VectorStore


In [47]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_document",persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
        except Exception as e:
            print(f"Error initializing vector store {e}")
            raise
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding{len(documents)}documents to vector store")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"successfully added {len(documents)} to vector store")
            print(f"Total documents in collections:{self.collection.count()}")
        except Exception as e:
            print(f"failed to add to vector db:{e}")
            raise
vectorstore=VectorStore()
vectorstore


In [33]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T04:54:38+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T04:54:38+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\ai_rag_notes.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_rag_notes.pdf', 'file_type': 'pdf'}, page_content='Ai Rag Notes\nArtificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems that can perform\ntasks associated with human intelligence, such as reasoning, learning, perception, and language\nunderstanding.\nMachine Learning\nMachine Learning is a subset of AI in which algorithms learn patterns from data and use those patterns to\nmake predictions or decisions.\nGenerative AI\nGenerative AI systems create new content such as text, images, audio, video, and code. Large language\

In [48]:
###Convert text to embeddings
texts=[doc.page_content for doc in chunks]

###Generate the Embeddings
embeddings=embedding_manager.generate_embedding(texts)

###store in the vector database
vectorstore.add_documents(chunks,embeddings)


Generating embedding for 3 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.06it/s]

Generated embeddings with shape (3, 384)
Adding3documents to vector store


successfully added 3 to vector store
Total documents in collections:3
